# 00 - Data extraction: Heart Disease UCI -> CSV
This notebook is self-contained. It gets the **UCI Heart Disease** data, checks that it is the right dataset, and writes the CSV files used by the rest of the project.

**Which data is this?** The UCI repository holds hundreds of datasets. The one the assignment asks for is *Heart Disease* (UCI id 45): four hospital databases (Cleveland, Hungary, Switzerland, VA Long Beach), each with 76 raw attributes of which 14 are used in all published work. The dataset's own documentation says the Cleveland database is the one ML researchers have actually used, so the modelling uses **Cleveland (303 patients, 13 features + diagnosis)**. The other three sites are also exported for reference only.

**Outputs**
| File | Content |
|---|---|
| `data/raw/heart_disease_cleveland_raw.csv` | Cleveland data with column names, original 0-4 label, `?` -> empty |
| `data/processed/heart_clean.csv` | cleaned modelling file (binary `target`), used by training |
| `data/raw/heart_disease_all_sites_raw.csv` | all four sites stacked, with a `site` column (reference only) |

In [1]:
from pathlib import Path
import io
import zipfile

import pandas as pd
import requests

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Official UCI locations (dataset id 45)
UCI_DIR_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/45/heart+disease.zip"
LOCAL_ZIP = RAW_DIR / "heart_disease.zip"     # offline copy (the zip from the assignment link)

COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
SITES = {
    "cleveland": "processed.cleveland.data",
    "hungarian": "processed.hungarian.data",
    "switzerland": "processed.switzerland.data",
    "va_long_beach": "processed.va.data",
}
EXPECTED_ROWS = {"cleveland": 303, "hungarian": 294, "switzerland": 123, "va_long_beach": 200}

## 1. Fetch the four processed files
Order of attempts: (1) download each file from UCI, (2) otherwise read from the local zip `data/raw/heart_disease.zip`. Either way the content is the same UCI files.

In [2]:
def fetch_from_uci(timeout=15) -> dict:
    out = {}
    for site, fname in SITES.items():
        r = requests.get(UCI_DIR_URL + fname, timeout=timeout)
        r.raise_for_status()
        out[site] = r.text
    return out


def fetch_from_zip(zip_path: Path) -> dict:
    with zipfile.ZipFile(zip_path) as zf:
        return {site: zf.read(fname).decode("latin-1") for site, fname in SITES.items()}


try:
    texts = fetch_from_uci()
    source = "UCI download"
except Exception as exc:                      # no internet, proxy, UCI down ...
    print(f"Download failed ({type(exc).__name__}); falling back to {LOCAL_ZIP.name}")
    texts = fetch_from_zip(LOCAL_ZIP)
    source = f"local zip ({LOCAL_ZIP.name})"
print("data source:", source)

Download failed (ConnectionError); falling back to heart_disease.zip
data source: local zip (heart_disease.zip)


## 2. Parse and validate
Missing values are written as `?` in these files. The checks below fail loudly if the files are not the heart-disease data (wrong row counts, wrong number of columns, label outside 0-4).

In [3]:
frames = {}
for site, text in texts.items():
    df = pd.read_csv(io.StringIO(text), header=None, names=COLUMNS, na_values="?")
    assert df.shape[1] == 14, f"{site}: expected 14 columns, got {df.shape[1]}"
    assert len(df) == EXPECTED_ROWS[site], f"{site}: expected {EXPECTED_ROWS[site]} rows, got {len(df)}"
    assert df["num"].dropna().isin([0, 1, 2, 3, 4]).all(), f"{site}: unexpected label values"
    frames[site] = df

summary = pd.DataFrame({
    "rows": {s: len(d) for s, d in frames.items()},
    "missing cells": {s: int(d.isna().sum().sum()) for s, d in frames.items()},
    "% with disease (num>0)": {s: round((d["num"] > 0).mean() * 100, 1) for s, d in frames.items()},
})
summary

,rows,missing cells,% with disease (num>0)
cleveland,303,6,45.9
hungarian,294,782,36.1
switzerland,123,273,93.5
va_long_beach,200,698,74.5


The four sites differ a lot: Hungarian and Swiss files have many missing values (e.g. 99% of `ca` and 90% of `thal` are missing in the Hungarian file), and the disease rate ranges from 36% to 94%. Mixing them would add site effects and heavy missingness, so the modelling uses the Cleveland file, which is the standard choice and the one the dataset documentation points to.

In [4]:
frames["hungarian"].isna().mean().round(2).to_frame("hungarian missing share").T

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
hungarian missing share,0.0,0.0,0.0,0.0,0.08,0.03,0.0,0.0,0.0,0.0,0.65,0.99,0.9,0.0


## 3. Write the CSV files

In [5]:
cleveland = frames["cleveland"]
cleveland.to_csv(RAW_DIR / "heart_disease_cleveland_raw.csv", index=False)

# Keep the original UCI text file as well: the training code reads it
(RAW_DIR / "processed.cleveland.data").write_text(texts["cleveland"])

all_sites = pd.concat([d.assign(site=s) for s, d in frames.items()], ignore_index=True)
all_sites.to_csv(RAW_DIR / "heart_disease_all_sites_raw.csv", index=False)
print("raw cleveland:", cleveland.shape, "| all sites:", all_sites.shape)

raw cleveland: (303, 14) | all sites: (920, 15)


## 4. Clean the Cleveland data -> `heart_clean.csv`
* label `num` (0-4) -> binary `target` (0 = no disease, 1 = disease)
* drop duplicate rows
* categorical/binary codes cast to float
* the 6 missing values (`ca`: 4, `thal`: 2) are **kept** as NaN on purpose: they are imputed inside the model pipeline using training data only, to avoid leakage

In [6]:
clean = cleveland.copy()
clean["target"] = (clean["num"] > 0).astype(int)
clean = clean.drop(columns=["num"]).drop_duplicates().reset_index(drop=True)
for col in ["cp", "restecg", "slope", "thal", "ca", "sex", "fbs", "exang"]:
    clean[col] = clean[col].astype("float64")

clean.to_csv(PROC_DIR / "heart_clean.csv", index=False)

print("shape:", clean.shape)
print("class balance:", clean["target"].value_counts().to_dict())
print("missing values:", clean.isna().sum()[clean.isna().sum() > 0].to_dict())
clean.head()

shape: (303, 14)
class balance: {0: 164, 1: 139}
missing values: {'ca': 4, 'thal': 2}


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 5. Final sanity check
Re-read the saved file the way the training code does.

In [7]:
check = pd.read_csv(PROC_DIR / "heart_clean.csv")
assert check.shape == (303, 14)
assert set(check["target"].unique()) == {0, 1}
assert list(check.columns) == COLUMNS[:-1] + ["target"]
print("OK -", check.shape[0], "patients,", check.shape[1] - 1, "features + target")
check.describe().T[["mean", "min", "max"]].round(2)

OK - 303 patients, 13 features + target


,mean,min,max
age,54.44,29.0,77.0
sex,0.68,0.0,1.0
cp,3.16,1.0,4.0
trestbps,131.69,94.0,200.0
chol,246.69,126.0,564.0
fbs,0.15,0.0,1.0
restecg,0.99,0.0,2.0
thalach,149.61,71.0,202.0
exang,0.33,0.0,1.0
oldpeak,1.04,0.0,6.2
